In [6]:
import os, sys
 
WORK  = '/kaggle/working'
REPO  = 'https://github.com/PhuongThao-2005/TULIP-MedML.git'
RNAME = 'TULIP-MedML'
 
if os.path.exists(f'{WORK}/{RNAME}'):
    os.system(f'cd {WORK}/{RNAME} && git pull')
    print('Pulled latest')
else:
    ret = os.system(f'cd {WORK} && git clone {REPO}')
    print('Cloned' if ret == 0 else 'Clone FAILED')
 
os.system('pip install pyyaml timm -q')
sys.path.insert(0, f'{WORK}/{RNAME}')
os.chdir(f'{WORK}/{RNAME}')
 
print('Working dir :', os.getcwd())
print('Repo contents:', os.listdir('.'))

Already up to date.
Pulled latest
Working dir : /kaggle/working/TULIP-MedML
Repo contents: ['notebooks', '.git', 'data', 'README.md', 'src', 'requirements.txt', '.gitignore']


In [8]:
# ── Cell 2: Kiểm tra GPU + dataset ───────────────────────────────────
import torch, pandas as pd
 
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
 
ROOT = '/kaggle/input/datasets/ashery/chexpert'
df   = pd.read_csv(f'{ROOT}/train.csv')
 
# Path trong CSV có dạng CheXpert-v1.0-small/... → bỏ phần đầu
sample = '/'.join(df['Path'].iloc[0].split('/')[1:])
abs_p  = os.path.join(ROOT, sample)
 
print('\nDataset root :', ROOT)
print('train.csv    :', len(df), 'rows')
print('val.csv      :', len(pd.read_csv(f'{ROOT}/valid.csv')), 'rows')
print('Image exists :', os.path.isfile(abs_p))
 
print('\nData files in repo:')
for fname in ['data/chexpert_glove_word2vec.npy', 'data/chexpert_adj.pkl']:
    exists = os.path.isfile(fname)
    size   = os.path.getsize(fname) / 1024 if exists else 0
    print(f'  {fname:45s} {"OK" if exists else "MISSING":8s} {size:.1f} KB')


GPU  : Tesla T4
VRAM : 15.6 GB

Dataset root : /kaggle/input/datasets/ashery/chexpert
train.csv    : 223414 rows
val.csv      : 234 rows
Image exists : True

Data files in repo:
  data/chexpert_glove_word2vec.npy              OK       16.5 KB
  data/chexpert_adj.pkl                         OK       1.0 KB


In [9]:
# ── Cell 3: Build adjacency matrix (nếu chưa có) ──────────────────────────────
# Co-occurrence được tính chỉ trên nhãn = 1 (positive).
# Uncertain (-1) và negative (0) đều không đếm là co-occur,
# để tránh inflate co-occurrence của các bệnh hiếm / không rõ.
import numpy as np, pickle
from src.data.chexpert import CHEXPERT_CLASSES
 
ADJ_PATH = 'data/chexpert_adj.pkl'
os.makedirs('data', exist_ok=True)
 
if not os.path.isfile(ADJ_PATH):
    print('Building adjacency matrix from train.csv ...')
    df_adj = pd.read_csv(f'{ROOT}/train.csv')
    labels = df_adj[CHEXPERT_CLASSES].fillna(0).values  # NaN → 0
 
    # Chỉ tính co-occur khi cả hai label == 1; uncertain (-1) → 0
    labels_bin = (labels == 1).astype(np.float32)       # {0, 1}
 
    n = len(CHEXPERT_CLASSES)
    adj  = np.zeros((n, n), dtype=np.float32)
    nums = np.zeros(n,      dtype=np.float32)
 
    for row in labels_bin:
        pos = np.where(row == 1)[0]
        nums[pos] += 1
        for i in pos:
            for j in pos:
                adj[i, j] += 1
 
    pickle.dump({'adj': adj, 'nums': nums}, open(ADJ_PATH, 'wb'))
    print(f'  Saved → {ADJ_PATH}  (shape {adj.shape})')
else:
    print(f'adj exists: {ADJ_PATH}')

adj exists: data/chexpert_adj.pkl


In [10]:
os.system(
    'python src/train.py '
    f'--config src/configs/test.yaml '
    '--subset 2000'
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/kaggle/working/TULIP-MedML/src/util.py:113: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  self.scores = torch.FloatTensor(torch.Float

Config: C1_smoke_test
[CheXpert] 44682 samples | policy=zeros
[CheXpert] 234 samples | policy=zeros
Subset mode: 200 train / 50 val
lr: [0.001 0.01 ]


Val:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch: [0][0/13]	Loss 166.6128 (166.6128)
Epoch: [0]	Loss 39.2866	mAP 17.378


Val: 100%|██████████| 4/4 [00:01<00:00,  2.86it/s]


Test: [0/4]	Loss 7.7184 (7.7184)

Val:	Loss 6.0411

Class                                    AUC
---------------------------------------------
No Finding                            0.5489
Enlarged Cardiomediastinum            0.6019
Cardiomegaly                          0.7125
Lung Opacity                          0.5000
Lung Lesion                              nan
Edema                                 0.6515
Consolidation                         0.6454
Pneumonia                             0.4583
Atelectasis                           0.5904
Pneumothorax                          0.3901
Pleural Effusion                      0.7156
Pleural Other                            nan
Fracture                                 nan
Support Devices                       0.4884
---------------------------------------------
mean_auc                              0.5730
map                                   0.2748
unc_auc                                  N/A
save model /kaggle/working/checkpoints/c1/chec

0

In [11]:
!cd /kaggle/working/TULIP-MedML && python src/train.py \
    --config src/configs/test.yaml \
    --subset 2000

Config: C1_smoke_test
[CheXpert] 44682 samples | policy=zeros
[CheXpert] 234 samples | policy=zeros
Subset mode: 2000 train / 222 val
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/kaggle/working/TULIP-MedML/src/util.py:113: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To 

In [13]:
!cd /kaggle/working/TULIP-MedML && python src/train.py \
    --config src/configs/test.yaml \
    --subset 10000

Config: C1_smoke_test
[CheXpert] 44682 samples | policy=zeros
[CheXpert] 234 samples | policy=zeros
Subset mode: 10000 train / 234 val
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/kaggle/working/TULIP-MedML/src/util.py:113: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To